# 00. Test the AS4777 curve keystone

This notebook validates "shared/as4777_curves.py" **offline** (no AWS, no data).

**What it checks**
1. The module imports and reads set-points from `ciccada_config.AS4777`.
2. The Volt-VAr, Volt-Watt and capability curves hit their known set-points.
3. The Python functions and the SQL-string generators agree (single source of truth).

In [ ]:
# Bootstrap: put the shared folder on the path
# EDIT THIS ONE LINE to the absolute path of your `shared/` folder.
import sys, pathlib
SHARED = pathlib.Path(r"../../shared")  # <-- EDIT
sys.path.insert(0, str(SHARED))

import as4777_curves as c
from ciccada_config import AS4777
print("Imported OK. Set-points:")
print("  VVAR:", AS4777["VVAR"])
print("  VW:  ", AS4777["VW"])
print("  TOL: ", AS4777["TOL_FRAC"])

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

## 1. Volt-VAr required-Q curve. Set points check

Expected (for S_rated = 5 kW):
- V ≤ 207 → +2.20 kvar (full supply, +0.44×5)
- V = 213.5 (midpoint) → +1.10 (half)
- V = 220~240 (deadband) → 0
- V = 249 (midpoint of 240–258) → −1.50 (half of -0.60×5)
- V ≥ 258 → -3.00 (full absorb)

In [ ]:
S = 5.0
for v in (200, 207, 213.5, 220, 240, 249, 258, 265):
    print(f"  V={v:6.1f}  ->  Q_required = {c.vvar_required_q(v, S):+7.3f} kvar")

In [ ]:
# Plot the full curve
V = np.linspace(200, 270, 400)
Q = [c.vvar_required_q(v, S) for v in V]
plt.figure(figsize=(8, 4))
plt.plot(V, Q, lw=2)
plt.axhline(0, color='k', lw=0.5)
for vx in (207, 220, 240, 258):
    plt.axvline(vx, ls=':', color='grey', lw=0.8)
plt.xlabel("Voltage (V)"); plt.ylabel("Required Q (kvar)")
plt.title("Volt-VAr required-Q curve (Australia A, S_rated = 5 kW)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 2. Volt-Watt max-P curve

Expected (S_rated = 5 kW): 100% (5.0) at/below 253 V, ramp to 20% (1.0) at 260 V.

In [ ]:
for v in (250, 253, 256.5, 260, 263):
    print(f"  V={v:6.1f}  ->  P_max = {c.vw_max_p(v, S):6.3f} kW")

## 3. Capability curve (R10)

Max reactive the inverter *can* absorb given current P. Note the S-circle:
as P approaches S_rated, the reactive headroom shrinks to zero.

In [ ]:
for p in (0.5, 2.0, 3.5, 4.5, 5.0):
    print(f"  P={p:5.2f}  ->  Q_cap_absorbing = {c.q_cap_absorbing(p, S):+7.3f} kvar")

print()
print("R10 demonstration — a site where S_99 (6.0) exceeds nameplate (5.0):")
print("  At P = 5.5 kW (above nameplate):")
print(f"    Using nameplate 5.0: Q_cap = {c.q_cap_absorbing(5.5, 5.0):+.3f}  (clamped to 0 — WRONG)")
print(f"    Using S_99      6.0: Q_cap = {c.q_cap_absorbing(5.5, 6.0):+.3f}  (physically correct)")

## 4. Python vs SQL agreement (single source of truth)

The SQL generators must produce expressions that evaluate to the same numbers
as the Python functions. Here we just print the SQL so you can eyeball it; the
real end-to-end check happens when the Stage-1 builders run against Athena.

In [ ]:
print("Volt-VAr SQL fragment:")
print(c.vvar_required_q_sql("V", "ac_capacity_kw"))
print("\nVolt-Watt SQL fragment:")
print(c.vw_max_p_sql("V", "ac_capacity_kw"))
print("\nCapability SQL fragment (R10: pass s_99):")
print(c.q_cap_absorbing_sql("P_kW", "s_99"))

## Done

If every set-point above matches expectation, the keystone is validated. 

Proceed to `10_run_stage1_pipeline.ipynb`.